# Chapter 10 · Experimental and Quasi-Experimental Designs in Health: companion notebook
*Data Science and Artificial Intelligence in Health, Volume 1*

> **This notebook accompanies the book chapter; it does not reproduce it.** Definitions, derivations, and explanations are in the book. Here you run the calculations on the case and check the numbers. All data are **synthetic**, and the values are for teaching purposes only; they have no correlation with the real world.

**Part A** follows the chapter's case, the record of a hypertension service with ten outpatient clinics, through the director's seven questions. **Part B** gives the code and result of each chapter exercise that needs code (statements are in the book). Run the cells in order.

## Part A. The case

**Data.** `data/clinic_patients.csv`: 6,744 rows, one per adult starting treatment over 24 months, and ten columns: `patient_id`, `clinic`, `month`, `period`, `clinic_start_month`, `diabetes`, `baseline_sbp`, `allocated`, `received`, `uncontrolled_12w`. Each cell carries the number of the book's section (10.1, 10.2, ...), so you can follow the chapter and the notebook side by side.

### 10.1 Introduction: the data

In [1]:
# 1. Read the record: one row is one adult
import pandas as pd
from math import sqrt, log, exp
from statistics import NormalDist
from scipy.stats import t

df = pd.read_csv("data/clinic_patients.csv")

# 2. Count rows and columns
rows, columns = df.shape
print("The record has", rows, "rows (one per adult) and", columns, "columns.")

# 3. A True/False column that is True when the pressure is still uncontrolled at 12 weeks (used from here on)
df["y"] = df["uncontrolled_12w"] == "yes"
df.head()                                      # The first five rows

The record has 6744 rows (one per adult) and 10 columns.


,patient_id,clinic,month,period,clinic_start_month,diabetes,baseline_sbp,allocated,received,uncontrolled_12w,y
0,A00001,C01,1,before,19,no,149,separate tablets,no,no,False
1,A00002,C01,1,before,19,no,144,separate tablets,no,no,False
2,A00003,C01,1,before,19,no,159,separate tablets,no,no,False
3,A00004,C01,1,before,19,no,150,separate tablets,no,yes,True
4,A00005,C01,1,before,19,no,155,separate tablets,no,yes,True


### 10.2 The comparison and the outcome risk

*What exactly is compared, and what is the risk of uncontrolled pressure in the trial?*

In [2]:
# 1. The trial months are the rows with period "trial" (months 7-12)
trial = df[df["period"] == "trial"]

# 2. Count the adults and those with uncontrolled pressure at 12 weeks
n_trial = len(trial)
events = trial["y"].sum()

# 3. Divide the count by all the adults of the trial
print("Adults in the trial months:", n_trial)
print("Uncontrolled at 12 weeks:", events, "of", n_trial, "=", round(100 * events / n_trial, 1), "%")
print("Overall risk:", round(events / n_trial, 4))

Adults in the trial months: 1716
Uncontrolled at 12 weeks: 478 of 1716 = 27.9 %
Overall risk: 0.2786


### 10.3.1 Randomization and baseline balance

*How can the two groups be made comparable and kept so?*

In [3]:
# 1. Take each assigned arm and count its adults
for arm in ["single pill", "separate tablets"]:
    group = trial[trial["allocated"] == arm]
    n = len(group)

    # 2. Count the adults with baseline systolic pressure of at least 160 mmHg
    high = (group["baseline_sbp"] >= 160).sum()

    # 3. Count the adults with diabetes
    diabetes = (group["diabetes"] == "yes").sum()
    print(arm + ":", n, "adults;", high, "with baseline pressure of at least 160 mmHg =", round(100 * high / n, 1), "%;",
          diabetes, "with diabetes =", round(100 * diabetes / n, 1), "%")

single pill: 857 adults; 341 with baseline pressure of at least 160 mmHg = 39.8 %; 223 with diabetes = 26.0 %
separate tablets: 859 adults; 331 with baseline pressure of at least 160 mmHg = 38.5 %; 207 with diabetes = 24.1 %


### 10.4.2 Intention to treat and uncertainty

*How is the trial analyzed, and what does its difference mean?*

In [4]:
# 1. In each assigned arm, count the adults and those uncontrolled
single = trial[trial["allocated"] == "single pill"]
separate = trial[trial["allocated"] == "separate tablets"]
a, n = single["y"].sum(), len(single)
b, m = separate["y"].sum(), len(separate)
print("Single pill:", a, "of", n)
print("Separate tablets:", b, "of", m)

# 2. Risk = uncontrolled adults / all adults of the arm
p = a / n
q = b / m

# 3. Risk difference (percentage points) and risk ratio
d = 100 * (p - q)
print("Risk difference:", round(d, 1), "percentage points")
print("Risk ratio:", round(p / q, 2))

# 4. Standard error and 95% interval of the difference (normal approximation)
z = NormalDist().inv_cdf(0.975)
se = 100 * sqrt(p * (1 - p) / n + q * (1 - q) / m)
itt_lower = d - z * se
itt_upper = d + z * se
print("Standard error:", round(se, 4), "; normal multiplier:", round(z, 4))
print("Difference:", round(d, 1), "; 95% CI", round(itt_lower, 1), "to", round(itt_upper, 1))

# 5. The ratio is handled on the logarithm scale, then back with exp
se_log = sqrt(1 / a - 1 / n + 1 / b - 1 / m)
print("Ratio:", round(p / q, 2), "; 95% CI", round(exp(log(p / q) - z * se_log), 2), "to", round(exp(log(p / q) + z * se_log), 2))

Single pill: 199 of 857
Separate tablets: 279 of 859
Risk difference: -9.3 percentage points
Risk ratio: 0.71
Standard error: 2.1525 ; normal multiplier: 1.96
Difference: -9.3 ; 95% CI -13.5 to -5.0
Ratio: 0.71 ; 95% CI 0.61 to 0.84


### 10.4.3 Per-protocol questions and treatment received

In [5]:
# 1. Keep only the adults whose treatment received matches the assigned arm
matches = (trial["allocated"] == "single pill") == (trial["received"] == "yes")
kept = trial[matches]

# 2. In this restricted group, count events and adults in each arm
single = kept[kept["allocated"] == "single pill"]
separate = kept[kept["allocated"] == "separate tablets"]
a, n = single["y"].sum(), len(single)
b, m = separate["y"].sum(), len(separate)
print("Retained adults:", len(kept), "; events/total:", a, "/", n, ",", b, "/", m)

# 3. Restricted risk ratio and its 95% interval (logarithm scale)
ratio = (a / n) / (b / m)
se_log = sqrt(1 / a - 1 / n + 1 / b - 1 / m)
print("Restricted ratio:", round(ratio, 2), "; 95% CI", round(exp(log(ratio) - z * se_log), 2), "to", round(exp(log(ratio) + z * se_log), 2))

Retained adults: 1610 ; events/total: 169 / 782 , 270 / 828
Restricted ratio: 0.66 ; 95% CI 0.56 to 0.78


### 10.5.3 Superiority, noninferiority, and equivalence

*Which other designs could have answered the director's question?*

In [6]:
# 1. Use the interval of the intention-to-treat analysis (section 10.4.2)
print("Upper limit of the 95% interval:", round(itt_upper, 1), "percentage points")

# 2. Superiority: the whole interval lies below zero
print("Superiority (upper limit below zero):", itt_upper < 0)

# 3. Noninferiority: the upper limit lies below the margin (illustrative margins)
for margin in [2, 5, 10]:
    print("Noninferiority with margin", margin, "points:", itt_upper < margin)

# 4. Equivalence: the interval lies inside the margin on both sides (illustrative margins)
for margin in [5, 10, 15]:
    print("Equivalence with margin", margin, "points:", (itt_lower > -margin) and (itt_upper < margin))

Upper limit of the 95% interval: -5.0 percentage points
Superiority (upper limit below zero): True
Noninferiority with margin 2 points: True
Noninferiority with margin 5 points: True
Noninferiority with margin 10 points: True
Equivalence with margin 5 points: False
Equivalence with margin 10 points: False
Equivalence with margin 15 points: True


### 10.6.1 Cluster-randomized trials

*Can the clinics themselves be assigned at random, even in steps?*

In [7]:
# 1. Months 13-18: the rows of the clinics that adopted at random in month 13 or stayed on separate tablets
rollout = df[df["month"].between(13, 18)]

# 2. Pooled by patient: events and adults in each arm
for arm in ["single pill", "separate tablets"]:
    group = rollout[rollout["allocated"] == arm]
    print(arm + ":", group["clinic"].nunique(), "clinics;", group["y"].sum(), "of", len(group))
single = rollout[rollout["allocated"] == "single pill"]
separate = rollout[rollout["allocated"] == "separate tablets"]
print("Pooled risk ratio:", round((single["y"].sum() / len(single)) / (separate["y"].sum() / len(separate)), 2))

# 3. Risk of each clinic, then the average of the clinic risks in each arm
clinic_risk = rollout.groupby(["allocated", "clinic"])["y"].mean()
p_clinic = clinic_risk.loc["single pill"]
q_clinic = clinic_risk.loc["separate tablets"]
print("Mean clinic risks:", round(p_clinic.mean(), 4), ",", round(q_clinic.mean(), 4))

# 4. Welch interval for the difference of the clinic averages (each clinic is one observation)
u = p_clinic.var() / len(p_clinic)
v = q_clinic.var() / len(q_clinic)
dof = (u + v) ** 2 / (u * u / (len(p_clinic) - 1) + v * v / (len(q_clinic) - 1))
d_clinic = 100 * (p_clinic.mean() - q_clinic.mean())
se_clinic = 100 * sqrt(u + v)
cut = t.ppf(0.975, dof)
print("Standard error:", round(se_clinic, 4), "; degrees of freedom:", round(dof, 2), "; t multiplier:", round(cut, 4))
print("Difference:", round(d_clinic, 1), "; 95% CI", round(d_clinic - cut * se_clinic, 1), "to", round(d_clinic + cut * se_clinic, 1))

single pill: 5 clinics; 179 of 819
separate tablets: 5 clinics; 236 of 852
Pooled risk ratio: 0.79
Mean clinic risks: 0.2176 , 0.2776
Standard error: 1.6944 ; degrees of freedom: 8.0 ; t multiplier: 2.3061
Difference: -6.0 ; 95% CI -9.9 to -2.1


### 10.6.2 Stepped-wedge cluster trials

In [8]:
# 1. Risk in each group of clinics and period (the clinics that switched in month 13 and in month 19)
early = df[df["clinic_start_month"] == 13]
late = df[df["clinic_start_month"] == 19]
early_first = early[early["month"].between(13, 18)]["y"].mean()
early_second = early[early["month"].between(19, 24)]["y"].mean()
late_first = late[late["month"].between(13, 18)]["y"].mean()
late_second = late[late["month"].between(19, 24)]["y"].mean()
print("Switched in month 13:", round(100 * early_first, 1), "%,", round(100 * early_second, 1), "%")
print("Switched in month 19:", round(100 * late_first, 1), "%,", round(100 * late_second, 1), "%")

# 2. Change in each group from months 13-18 to months 19-24
change_early = early_second - early_first
change_late = late_second - late_first
print("Changes:", round(100 * change_late, 1), ",", round(100 * change_early, 1), "points")

# 3. Stepped-wedge contrast: the change of the late clinics minus the change of the early clinics
print("Stepped-wedge contrast:", round(100 * (change_late - change_early), 1), "points")

Switched in month 13: 21.9 %, 19.5 %
Switched in month 19: 27.7 %, 21.0 %
Changes: -6.7 , -2.4 points
Stepped-wedge contrast: -4.3 points


### 10.7.1 Before-and-after comparisons and secular trends

*What can the records show without random assignment?*

In [9]:
# 1. The clinics that adopted in month 13: months 1-6 (before) and months 13-18 (after)
adopters = df[df["clinic_start_month"] == 13]
before = adopters[adopters["month"].between(1, 6)]
after = adopters[adopters["month"].between(13, 18)]

# 2. Pool the counts before dividing
risk_before = before["y"].sum() / len(before)
risk_after = after["y"].sum() / len(after)
print("Before:", before["y"].sum(), "/", len(before), "=", round(100 * risk_before, 1), "%")
print("After:", after["y"].sum(), "/", len(after), "=", round(100 * risk_after, 1), "%")
print("After/before ratio:", round(risk_after / risk_before, 2))

Before: 311 / 830 = 37.5 %
After: 179 / 819 = 21.9 %
After/before ratio: 0.58


### 10.7.2 Controlled comparisons and difference in differences

In [10]:
# 1. The four risks: adopting clinics (month 13) and comparison clinics (month 19), before and after
comparison = df[df["clinic_start_month"] == 19]
a0 = adopters[adopters["month"].between(1, 6)]["y"].mean()
a1 = adopters[adopters["month"].between(13, 18)]["y"].mean()
b0 = comparison[comparison["month"].between(1, 6)]["y"].mean()
b1 = comparison[comparison["month"].between(13, 18)]["y"].mean()
print("Adopting clinics:", round(100 * a0, 1), "%,", round(100 * a1, 1), "%")
print("Comparison clinics:", round(100 * b0, 1), "%,", round(100 * b1, 1), "%")

# 2. Difference in differences = change of the adopting clinics minus change of the comparison clinics
print("Changes:", round(100 * (a1 - a0), 1), ",", round(100 * (b1 - b0), 1), "points")
print("Difference in differences:", round(100 * ((a1 - a0) - (b1 - b0)), 1), "points")

# 3. The path assumed without adoption and the ratio of ratios
print("Assumed endpoint without adoption:", round(100 * (a0 + b1 - b0), 1), "%")
print("Ratio of ratios:", round((a1 / a0) / (b1 / b0), 2))

Adopting clinics: 37.5 %, 21.9 %
Comparison clinics: 35.7 %, 27.7 %
Changes: -15.6 , -8.0 points
Difference in differences: -7.6 points
Assumed endpoint without adoption: 29.5 %
Ratio of ratios: 0.75


### 10.7.3 Interrupted time series

In [11]:
# 1. Monthly counts for the clinics that adopted in month 13
monthly = adopters.groupby("month")["y"].agg(["sum", "size"])

# 2. Number of months before, during the trial, and after adoption
print("Months before the trial:", (monthly.index < 7).sum())
print("Trial months:", ((monthly.index >= 7) & (monthly.index <= 12)).sum())
print("Months after adoption:", (monthly.index >= 13).sum())

# 3. The first monthly risk, with its own denominator
first = monthly.iloc[0]
print("First month:", first["sum"], "/", first["size"], "=", round(100 * first["sum"] / first["size"], 1), "%")

Months before the trial: 6
Trial months: 6
Months after adoption: 12
First month: 54 / 134 = 40.3 %


## Part B. Exercise solutions

### 10.10 Exercise 1. Baseline balance

In [12]:
# 1. Take each assigned arm and count its adults
for arm in ["single pill", "separate tablets"]:
    group = trial[trial["allocated"] == arm]
    n = len(group)

    # 2. Count the adults with baseline systolic pressure of at least 160 mmHg
    high = (group["baseline_sbp"] >= 160).sum()

    # 3. Count the adults with diabetes
    diabetes = (group["diabetes"] == "yes").sum()
    print(arm + ":", n, "adults;", high, "with baseline pressure of at least 160 mmHg =", round(100 * high / n, 1), "%;",
          diabetes, "with diabetes =", round(100 * diabetes / n, 1), "%")

single pill: 857 adults; 341 with baseline pressure of at least 160 mmHg = 39.8 %; 223 with diabetes = 26.0 %
separate tablets: 859 adults; 331 with baseline pressure of at least 160 mmHg = 38.5 %; 207 with diabetes = 24.1 %


### 10.10 Exercise 3. Assignment and adherence questions

In [13]:
# 1. The risks, the difference, and the ratio with their intervals (as in section 10.4.2)
single = trial[trial["allocated"] == "single pill"]
separate = trial[trial["allocated"] == "separate tablets"]
a, n = single["y"].sum(), len(single)
b, m = separate["y"].sum(), len(separate)
p = a / n
q = b / m
d = 100 * (p - q)
se = 100 * sqrt(p * (1 - p) / n + q * (1 - q) / m)
se_log = sqrt(1 / a - 1 / n + 1 / b - 1 / m)
print("Risks:", round(p, 4), ",", round(q, 4))
print("Difference:", round(d, 1), "; 95% CI", round(d - z * se, 1), "to", round(d + z * se, 1))
print("Ratio:", round(p / q, 2), "; 95% CI", round(exp(log(p / q) - z * se_log), 2), "to", round(exp(log(p / q) + z * se_log), 2))

# 2. How many adults received what they were assigned
matches = (trial["allocated"] == "single pill") == (trial["received"] == "yes")
print("Receipt matches assignment in", matches.sum(), "of", len(trial), "adults")

Risks: 0.2322 , 0.3248
Difference: -9.3 ; 95% CI -13.5 to -5.0
Ratio: 0.71 ; 95% CI 0.61 to 0.84
Receipt matches assignment in 1610 of 1716 adults


### 10.10 Exercise 6. Clinics within the allocated groups

In [14]:
# 1. Each clinic in months 13-18: uncontrolled adults, adults, and risk
table = rollout.groupby(["allocated", "clinic"])["y"].agg(["sum", "size"])
table.columns = ["uncontrolled", "patients"]
table["risk (%)"] = round(100 * table["uncontrolled"] / table["patients"], 1)
display(table)

# 2. The Welch interval of the equal-clinic difference (as in section 10.6.1)
print("Difference:", round(d_clinic, 1), "; 95% CI", round(d_clinic - cut * se_clinic, 1), "to", round(d_clinic + cut * se_clinic, 1))

uncontrolled  patients  risk (%)
allocated        clinic                                  
separate tablets C01               50       167      29.9
                 C02               42       180      23.3
                 C05               50       167      29.9
                 C07               45       163      27.6
                 C10               49       175      28.0
single pill      C03               37       167      22.2
                 C04               35       171      20.5
                 C06               36       160      22.5
                 C08               27       148      18.2
                 C09               44       173      25.4

Difference: -6.0 ; 95% CI -9.9 to -2.1


### 10.10 Exercise 7. The whole-service change

In [15]:
# 1. Months 1-6 and months 19-24 for the whole service
periods = [("months 1-6", df[df["month"].between(1, 6)]), ("months 19-24", df[df["month"].between(19, 24)])]

# 2. In each period: adults, uncontrolled, and those who received the single pill
for name, rows_in_period in periods:
    n = len(rows_in_period)
    uncontrolled = rows_in_period["y"].sum()
    uptake = (rows_in_period["received"] == "yes").sum()
    print(name + ": uncontrolled", uncontrolled, "/", n, "=", round(100 * uncontrolled / n, 1), "%; uptake", uptake, "/", n, "=", round(100 * uptake / n, 1), "%")

months 1-6: uncontrolled 621 / 1698 = 36.6 %; uptake 56 / 1698 = 3.3 %
months 19-24: uncontrolled 336 / 1659 = 20.3 %; uptake 1527 / 1659 = 92.0 %


### 10.10 Exercise 8. The stepped-wedge contrast against the cluster comparison

In [16]:
# 1. The stepped-wedge contrast (section 10.6.2) and the cluster comparison of months 13-18 pooled by patient
print("Stepped-wedge contrast:", round(100 * (change_late - change_early), 1), "points")
print("Cluster comparison (pooled by patient):", round(100 * (early_first - late_first), 1), "points")

Stepped-wedge contrast: -4.3 points
Cluster comparison (pooled by patient): -5.8 points


---
*Synthetic data (seed 33, `generate_data.py`). Teaching values only; nothing here is a clinical recommendation or an estimate for a real service.*